# Обучение одиночного нейрона

`Neuron` самостоятельно хранит веса и смещение, выполняет прямой и
обратный проход и обновляет параметры. Это отдельная модель для
изучения одного нейрона. `Unit` выполняет другую роль: он показывает
один нейрон, параметры которого принадлежат матричному `Layer`.

На примере AND сначала вычислим градиенты вручную по шагам, затем
запустим цикл обучения и сравним ответы с таблицей истинности.

In [ ]:
import numpy as np

from open_neuron.activations import sigmoid
from open_neuron.losses import MSE
from open_neuron.neurons import Neuron
from open_neuron.optimizers import SGD

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [0], [0], [1]])

neuron = Neuron(
    weights=[0.45, -0.12],
    bias=1.0,
    activation=sigmoid,
    loss=MSE(),
    optimizer=SGD(learning_rate=0.7, momentum=0.3),
)

## Прямой и обратный проход

Для каждого примера нейрон вычисляет взвешенную сумму и применяет
сигмоиду. Затем MSE превращает остаток `A - y` в градиент потерь;
производная сигмоиды передаёт его к весам и смещению. Вызов
`backward` только вычисляет градиенты — параметры изменит `update`,
который вызывается циклом `fit`.

In [ ]:
initial_predictions = neuron.forward(X)
initial_loss = neuron.loss_function.evaluate_loss(y, initial_predictions)
error = neuron.loss_function.evaluate_error(y, initial_predictions)
neuron.backward(error)

print('Взвешенные суммы Z:', neuron.Z.ravel())
print('Выходы A:', initial_predictions.ravel())
print('Начальная MSE:', initial_loss)
print('Градиент весов:', neuron.gradient['weights'].ravel())
print('Градиент смещения:', neuron.gradient['bias'].ravel())

## Обучение и предсказание

`fit` повторяет прямой проход, вычисление градиентов и обновление
параметров. После обучения `predict` считает выходы без обновления.
Округление вероятностей до ближайшего целого показывает предсказанный
класс для этой задачи.

In [ ]:
neuron.fit(X, y, epochs=100, batch_size=1, shuffle=False)
predictions = neuron.predict(X)

print('Вероятности:', predictions.ravel())
print('Предсказанные классы:', np.round(predictions).astype(int).ravel())
print('Ожидаемые классы:    ', y.ravel())
print('Состояние нейрона после обучения:')
print(neuron)